<a href="https://colab.research.google.com/github/SamuSab00/Immunogold-Dots-Project/blob/main/notebooks/09_architettura_encoder_decoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Fase 0**: Import Generico

In [ ]:
import os
import sys
import importlib
from google.colab import drive

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print("Drive già montato, salto il mount.")

sys.path.append('/content/drive/MyDrive/progetto_immunogold/scripts')
importlib.invalidate_caches()

import common
importlib.reload(common)
from common import *

print(f"Device attivo: {device}")
print(paths)
log('Notebook avviato correttamente')

Mounted at /content/drive
Device attivo: cpu
{'raw_images': '/content/drive/MyDrive/progetto_immunogold/raw_images', 'annotations': '/content/drive/MyDrive/progetto_immunogold/annotations', 'patches_images': '/content/drive/MyDrive/progetto_immunogold/patches/images', 'patches_coordinates': '/content/drive/MyDrive/progetto_immunogold/patches/coordinates', 'patches_density': '/content/drive/MyDrive/progetto_immunogold/patches/density_maps', 'checkpoints': '/content/drive/MyDrive/progetto_immunogold/checkpoints', 'logs': '/content/drive/MyDrive/progetto_immunogold/logs'}
[2026-09-25 09:58] Notebook avviato correttamente


## **Architettura Encoder-Decoder (U-Net compatta)**

La U-Net si compone di 4 elementi principali:


1.   **Encoder**
2.   **Bottleneck**
3.   **Decoder**
4.   **Output finale**

L'encoder diminuisce la risoluzione dell'immagine catturandone i pattern. Il decoder ricostruisce l'immagine riportandola alla sua risoluzione originale combinando le informazioni accumulate e le immagini di partenza.



### **1.   Encoder**

Per costruire la struttura del singolo encoder si utilizzano due strumenti indipendenti: il ***blocco convoluzionale*** e il ***max pooling***.

**Convoluzione**

Partendo dalla patch 256x256 a 1 canale, l'encoder esegue prima il *blocco convoluzionale*:


*   **nn.Conv2d** costruisce il filtro convoluzionale, *canali_in* è il numero di canali dell'immagine in input (1 per immagine in scala di grigi), *canali_out* è il numero di pattern che vogliamo che l'algoritmo sia in grado di riconoscere in parallelo, *kernel_size* è la dimensione del filtro (se = 3, filtro 3x3) *padding=1* aggiunge artificialmente un bordo di zeri per evitare che l'operazione mangi il bordo dell'immagine (il filtro non può centrarsi sui valori del bordo senza questo accorgimento).

Si eseguono due convoluzioni consecutive:

1.   canali_in --> canali_out (cambia il numero di canali, da 1 a 16)
2.   canali_out --> canali_out (inizia a riconoscere pattern in maniera più dettagliata prima di passare al pooling)

**Pooling**

Secondo blocco indipendente dell'encoder: il suo compito è ridurre la dimensionalità dell'input.


*   Divide l'immagine in blocchetti 2x2 non sovrapposti e per ciascun blocchetto tiene il valore massimo tra i 4 pixel che lo compongono.
*   L'immagine viene dimezzata in entrambe le dimensioni (256x256 --> 128x128) mantenendo tutti e 16 i canali (il pooling non tocca la profondità, opera solo sulla dimensionalità).

Una volta fatto ciò, con **livello_encoder** si richiama il singolo encoder per intero (convoluzione + pooling) che restituisce in output, oltre che l'immagine ridotta (*x_ridotto*) anche *skip* ossia la fotografia dell'immagine ad alta risoluzione associata che servirà poi per il decoder.

Infine con **Encoder** creiamo la struttura a 3 livelli, fino ad arrivare a una risoluzione di 32x32 con 64 canali di profondità.
Il return fornisce anche i 3 skip salvati ai vari livelli dell'encoder, in ordine dal primo (quello a risoluzione più alta) fino al terzo (quello corrispondente al bottleneck).


In [ ]:
class BloccoConv(nn.Module):
    def __init__(self, canali_in, canali_out):
        super().__init__()
        self.conv1 = nn.Conv2d(canali_in, canali_out, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(canali_out, canali_out, kernel_size=3, padding=1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        return x


class LivelloEncoder(nn.Module):
    def __init__(self, canali_in, canali_out):
        super().__init__()
        self.blocco = BloccoConv(canali_in, canali_out)
        self.pool = nn.MaxPool2d(kernel_size=2)

    def forward(self, x):
        skip = self.blocco(x)        # alta risoluzione, da salvare per il decoder
        x_ridotto = self.pool(skip)  # bassa risoluzione, da passare al livello successivo
        return x_ridotto, skip


class Encoder(nn.Module):
    def __init__(self, canali_in=1, canali_base=16):
        super().__init__()
        self.livello1 = LivelloEncoder(canali_in, canali_base)             # 1  -> 16
        self.livello2 = LivelloEncoder(canali_base, canali_base * 2)       # 16 -> 32
        self.livello3 = LivelloEncoder(canali_base * 2, canali_base * 4)   # 32 -> 64

    def forward(self, x):
        x, skip1 = self.livello1(x)
        x, skip2 = self.livello2(x)
        x, skip3 = self.livello3(x)
        return x, [skip1, skip2, skip3]

### **2. Bottleneck**

Sul bottleneck applichiamo solamente la funzione *BloccoConv* perchè vogliamo modificare solamente il numero di canali e non la dimensionalità, siccome siamo al punto di massima compressione deliberata. (ad es. se siamo a 64 canali passiamo a 128).

In [ ]:
class Bottleneck(nn.Module):
    def __init__(self, canali_in, canali_out):
        super().__init__()
        self.blocco = BloccoConv(canali_in, canali_out)

    def forward(self, x):
        return self.blocco(x)

### **3. Decoder**

Il decoder viene costruito usando ***upsample***, ***riduci_canali*** e ***blocco*** dopo aver concatenato lo *skip* tramite ***torch.cat***.

*   **upsample**

E' il corrispettivo inverso del pooling, quindi va a raddoppiare la risoluzione anzichè a dimezzarla ricevendo in input *scale_factor=2*.
*   **riduci_canali**

E' un operazione di adattamento dell'immagine utile per concatenare l'immagine corrente con lo *skip*. (l'immagine al bottleneck ha 128 canali mentre lo skip3 ne ha 64, siccome *upsample* non tocca la profondità, è necessario questo adattamento per riportare l'immagine in proporzione con lo *skip*)

* **torch.cat**

Concatenazione dell'immagine corrente allo *skip* del livello encoder relativo (a partire da skip3 a scendere). *dim=1* specifica l'asse lungo il quale va concatenata l'immagine (ricordando l'ordine di struttura in PyTorch - batch, canali, altezza, larghezza - con *dim=1* andiamo a concatenare sulla profondità)

* **blocco**

Utilizza lo stesso *BloccoConv* precedente, ma questa volta lo fa per andare a fondere le informazioni correnti con quelle dello *skip* concatenato. La rete è libera di bilanciare le due fonti di informazioni tramite i pesi allenabili.



In [ ]:
class LivelloDecoder(nn.Module):
    def __init__(self, canali_in, canali_out):
        super().__init__()
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.riduci_canali = nn.Conv2d(canali_in, canali_out, kernel_size=1)
        self.blocco = BloccoConv(canali_out * 2, canali_out)  # *2: dopo la concatenazione con lo skip

    def forward(self, x, skip):
        x = self.upsample(x)
        x = self.riduci_canali(x)
        x = torch.cat([x, skip], dim=1)
        x = self.blocco(x)
        return x


class Decoder(nn.Module):
    def __init__(self, canali_base=16):
        super().__init__()
        self.livello1 = LivelloDecoder(canali_base * 8, canali_base * 4)  # 128 -> 64
        self.livello2 = LivelloDecoder(canali_base * 4, canali_base * 2)  # 64  -> 32
        self.livello3 = LivelloDecoder(canali_base * 2, canali_base)      # 32  -> 16

    def forward(self, x, skips):
        skip1, skip2, skip3 = skips  # ordine di salvataggio: superficiale -> profondo
        x = self.livello1(x, skip3)  # il decoder incontra prima il più profondo/piccolo
        x = self.livello2(x, skip2)
        x = self.livello3(x, skip1)  # e finisce con il più superficiale/grande
        return x

### **4. Output Finale**

In questa fase abbiamo già raggiunto la dimensionalità desiderata (256x256), dobbiamo solamente andare a comprimere la profondità (16 --> 1) con un ultima operazione di convoluzione (*Conv2d*).

In [ ]:
class LivelloOutput(nn.Module):
    def __init__(self, canali_in):
        super().__init__()
        self.conv_finale = nn.Conv2d(canali_in, 1, kernel_size=1)
        self.attivazione = nn.Softplus()

    def forward(self, x):
        x = self.conv_finale(x)
        x = self.attivazione(x)
        return x

### **Struttura completa della U-Net**

Uniamo nella classe *UNetImmunogold* le 4 fasi spiegate in precedenza con i vari input che devono ricevere, e sviluppiamo la sequenza completa in *forward*, dove viene elaborato l'input x (immagine di partenza processata).

In [ ]:
class UNetImmunogold(nn.Module):
    def __init__(self, canali_in=1, canali_base=16):
        super().__init__()
        self.encoder = Encoder(canali_in=canali_in, canali_base=canali_base)
        self.bottleneck = Bottleneck(canali_in=canali_base * 4, canali_out=canali_base * 8)
        self.decoder = Decoder(canali_base=canali_base)
        self.output = LivelloOutput(canali_in=canali_base)

    def forward(self, x):
        x, skips = self.encoder(x)
        x = self.bottleneck(x)
        x = self.decoder(x, skips)
        x = self.output(x)
        return x

In [ ]:
modello = UNetImmunogold(canali_in=1, canali_base=16).to(device)

input_finto = torch.randn(1, 1, 256, 256).to(device)
output_finale = modello(input_finto)

print("Input:", input_finto.shape)
print("Output:", output_finale.shape)
print("Min output:", output_finale.min().item(), "(atteso >= 0)")

n_parametri = sum(p.numel() for p in modello.parameters())
print(f"Parametri allenabili: {n_parametri:,}")

Input: torch.Size([1, 1, 256, 256])
Output: torch.Size([1, 1, 256, 256])
Min output: 0.5944355130195618 (atteso >= 0)
Parametri allenabili: 449,489
